# Onsen Robot — Exploring Raw Robot Data

Connects to the running stack through **rosbridge** (`ws://localhost:9090`) and inspects the raw streams a data scientist works with: LIDAR scans, odometry vs ground truth, camera frames and arm/base command telemetry.

Prerequisites: `docker compose up` running, plus `pip install websocket-client numpy matplotlib pillow`.

In [ ]:
import base64, io, json, threading, time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import websocket

ROSBRIDGE = 'ws://localhost:9090'
latest = {}

def on_message(_ws, raw):
    msg = json.loads(raw)
    if msg.get('op') == 'publish':
        latest[msg['topic']] = msg['msg']

ws = websocket.WebSocketApp(ROSBRIDGE, on_message=on_message)
threading.Thread(target=ws.run_forever, daemon=True).start()
time.sleep(1)

def subscribe(topic, msg_type, throttle_ms=200):
    ws.send(json.dumps({'op': 'subscribe', 'topic': topic, 'type': msg_type,
                        'throttle_rate': throttle_ms}))

subscribe('/scan', 'sensor_msgs/LaserScan')
subscribe('/odom', 'nav_msgs/Odometry')
subscribe('/ground_truth/pose', 'geometry_msgs/PoseStamped')
subscribe('/camera/front/image_raw/compressed', 'sensor_msgs/CompressedImage', 500)
subscribe('/imu', 'sensor_msgs/Imu')
subscribe('/arm/state', 'std_msgs/String')
subscribe('/base/state', 'std_msgs/String')
time.sleep(3)
sorted(latest.keys())

## LIDAR scan — polar plot
Real raycasts against the physics world, including noise, dropouts and steam degradation near the baths.

In [ ]:
scan = latest['/scan']
ranges = np.array(scan['ranges'], dtype=float)
angles = scan['angle_min'] + np.arange(len(ranges)) * scan['angle_increment']
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(projection='polar')
ax.scatter(angles[ranges < scan['range_max']], ranges[ranges < scan['range_max']], s=2)
ax.set_title('LIDAR 360 (m)')
plt.show()

## Camera frame
JPEG-compressed offscreen render from the robot's front mount.

In [ ]:
frame = latest['/camera/front/image_raw/compressed']
img = Image.open(io.BytesIO(base64.b64decode(frame['data'])))
plt.figure(figsize=(8, 6)); plt.imshow(img); plt.axis('off'); plt.show()

## Odometry drift vs ground truth
Encoder-integrated odometry drifts during skid turns by construction. Collect both for ~20 s while driving the robot (or while the mission executor runs).

In [ ]:
odo, gt = [], []
for _ in range(100):
    if '/odom' in latest:
        p = latest['/odom']['pose']['pose']['position']; odo.append((p['x'], p['y']))
    if '/ground_truth/pose' in latest:
        p = latest['/ground_truth/pose']['pose']['position']; gt.append((p['x'], p['y']))
    time.sleep(0.2)
odo, gt = np.array(odo), np.array(gt)
plt.figure(figsize=(7, 5))
if len(odo): plt.plot(odo[:, 0], odo[:, 1], label='odom (encoders)')
if len(gt): plt.plot(gt[:, 0], gt[:, 1], label='ground truth')
plt.legend(); plt.axis('equal'); plt.title('Odometry drift'); plt.show()

## Controller telemetry
Arm firmware and base controller state as JSON — the same data the AI worker and safety aggregator consume.

In [ ]:
print('ARM :', json.dumps(json.loads(latest['/arm/state']['data']), indent=2))
print('BASE:', json.dumps(json.loads(latest['/base/state']['data']), indent=2))